In [ ]:
# ========== 安装依赖并导入：第 5 周 RAG（FAISS + OpenRouter）==========

# 用 %pip 安装本练习需要的 LangChain / 向量库 / 可视化 / Gradio 等包（IPython magic）
%pip install langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-openai plotly scikit-learn gradio python-dotenv faiss-cpu
# 安装完成后给一个可见确认
print("✅ Setup complete")

# 标准库与数值/可视化：os/glob/time；numpy；plotly 图；子图工具
import os, glob, time, numpy as np, plotly.graph_objects as go
# make_subplots：后面性能测试画双柱状图用
from plotly.subplots import make_subplots
# Path：路径对象（本格主要用字符串路径，仍一并导入）
from pathlib import Path
# load_dotenv：从 .env 读入环境变量（Environment Variables），避免密钥写进代码
from dotenv import load_dotenv
# TSNE：把高维 embedding 压到 2D，方便散点可视化
from sklearn.manifold import TSNE
# Gradio：快速搭聊天 UI
import gradio as gr
# shutil/tempfile：工具库（本笔记本后续未必用到，保持原导入）
import shutil, tempfile
# Counter：计数工具（保持原导入）
from collections import Counter

# ---------- LangChain 生态：文档加载 / 切分 / 向量库 / 本地嵌入 ----------
# DirectoryLoader + TextLoader：按目录批量读 Markdown
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# RecursiveCharacterTextSplitter：按字符递归切块（chunk）
from langchain_text_splitters import RecursiveCharacterTextSplitter
# FAISS：本地向量索引，避免某些环境下 Chroma 只读问题
from langchain_community.vectorstores import FAISS
# HuggingFaceEmbeddings：本地句向量模型（Sentence Transformers）
from langchain_huggingface import HuggingFaceEmbeddings
# OpenAI 客户端：这里会指向 OpenRouter 的兼容 API
from openai import OpenAI

# 导入完成确认
print("✅ Imports done")


In [ ]:
# ========== OpenRouter 客户端与模型名 ==========

# 加载 .env 里的密钥到进程环境变量
load_dotenv()
# 读取 OPENROUTER_API_KEY：优先 os.getenv，失败再试 os.environ.get（两路兜底）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY") or os.environ.get("OPENROUTER_API_KEY")
# 构造 OpenAI 兼容客户端：base_url 指向 OpenRouter，api_key 用上面读到的密钥
openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
# 选用的聊天模型 id（OpenRouter 路由名，勿改字符串）
MODEL = "openai/gpt-4o-mini"
# 打印当前走哪条模型，便于排查配置
print(f"✅ Using OpenRouter with {MODEL}")


In [ ]:
# ========== 定位或创建知识库目录（Insurellm 样本）==========

# 默认知识库相对路径名
KNOWLEDGE_BASE = "knowledge-base"
# 若当前目录没有该文件夹，再尝试课程仓库里常见的相对路径
if not os.path.isdir(KNOWLEDGE_BASE):
    # 两个候选：从 community-contributions 往上找 week5，或本地 week5
    for candidate in ["../../../week5/knowledge-base", "week5/knowledge-base"]:
        # 找到第一个存在的目录就采用并跳出
        if os.path.isdir(candidate):
            KNOWLEDGE_BASE = candidate
            break

# 如果仍然没有知识库：现场造最小样本，保证后面 RAG 能跑通
if not os.path.isdir(KNOWLEDGE_BASE):
    # 员工 / 产品两个子目录
    os.makedirs(f"{KNOWLEDGE_BASE}/employees", exist_ok=True)
    os.makedirs(f"{KNOWLEDGE_BASE}/products", exist_ok=True)

    # 写入员工样例 Markdown（内容字符串保持英文原样）
    with open(f"{KNOWLEDGE_BASE}/employees/alex_lancaster.md", 'w') as f:
        f.write("# Alex Lancaster\nCTO at Insurellm. Graduated from Manchester University.")
    # 写入产品样例 CarLLM
    with open(f"{KNOWLEDGE_BASE}/products/carllm.md", 'w') as f:
        f.write("# CarLLM\nAI auto insurance assistant.")
    # 写入产品样例 HomeLLM
    with open(f"{KNOWLEDGE_BASE}/products/homellm.md", 'w') as f:
        f.write("# HomeLLM\nAI home insurance assistant.")

# 递归列出知识库下所有 .md，确认文件数
files = glob.glob(f"{KNOWLEDGE_BASE}/**/*.md", recursive=True)
print(f"📁 Found {len(files)} files")


In [ ]:
# ========== 加载 Markdown 文档并切成重叠块（chunks）==========

# 收集所有 LangChain Document
documents = []
# 遍历知识库一级子目录（employees / products 等）
for folder in glob.glob(f"{KNOWLEDGE_BASE}/*"):
    # 只处理目录，跳过散落文件
    if os.path.isdir(folder):
        # DirectoryLoader：该文件夹下所有 *.md，用 TextLoader 读纯文本
        loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader)
        # 真正读入磁盘文件 → Document 列表
        docs = loader.load()
        # 给每篇文档打上 type=文件夹名，供后面着色/过滤
        for doc in docs:
            doc.metadata["type"] = os.path.basename(folder)
        # 并入总列表
        documents.extend(docs)

# 递归字符切分器：块长 500，重叠 100，减轻边界信息丢失
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# 对全部文档执行切分，得到检索粒度的 chunks
chunks = splitter.split_documents(documents)
print(f"🔪 Created {len(chunks)} chunks")


In [ ]:
# ========== 本地嵌入 + FAISS 向量库，并抽样向量供可视化 ==========

# 提示：开始算 embedding（可能稍慢）
print("🔄 Creating embeddings...")
# FAISS 向量存储（本格再次显式导入，便于单独重跑）
from langchain_community.vectorstores import FAISS
# HuggingFace 嵌入封装
from langchain_huggingface import HuggingFaceEmbeddings
# numpy：把向量列表转成数组，方便 TSNE
import numpy as np

# 轻量句向量模型 all-MiniLM-L6-v2（本地跑，不走 OpenAI Embedding API）
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 用全部 chunks 建 FAISS 索引（相似度检索的核心）
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Created FAISS vectorstore with {len(chunks)} chunks")

# 可视化只采样前 50 条，避免 TSNE 太慢
sample_size = min(50, len(chunks))
sample_chunks = chunks[:sample_size]
# 对每个样本文本再 embed_query，得到向量矩阵
vectors = np.array([embeddings.embed_query(c.page_content) for c in sample_chunks])
# 同步抽出 type/source，供散点图按类别着色
metadata = [{'type': c.metadata.get('type', 'unknown'), 'source': c.metadata.get('source', '')} for c in sample_chunks]
print(f"📊 Got {len(vectors)} vectors for visualization")


In [ ]:
# ========== t-SNE 降维 + Plotly 散点：看文档向量是否按类别聚簇 ==========

# 至少 2 个点才有意义做 2D 投影
if len(vectors) > 1:
    # perplexity 不能超过样本量-1；random_state 固定结果可复现
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    # 高维 → 二维坐标
    vectors_2d = tsne.fit_transform(vectors)

    # 空 Figure，后面按类别 add_trace
    fig = go.Figure()
    # 类别 → 颜色映射（与 metadata['type'] 对齐）
    colors = {'products':'blue', 'employees':'green', 'contracts':'red', 'company':'orange'}
    # 逐类筛选并画散点
    for cat, color in colors.items():
        # 布尔掩码：该点是否属于当前类别
        mask = [m['type']==cat for m in metadata]
        # 该类至少有一个点才画
        if any(mask):
            fig.add_trace(go.Scatter(x=vectors_2d[mask,0], y=vectors_2d[mask,1], mode='markers',
                                    name=cat, marker=dict(color=color)))
    # 布局：标题与画布大小
    fig.update_layout(title="📊 Document Vectors", width=800, height=500)
    # 在 Notebook 中展示交互图
    fig.show()


In [ ]:
# ========== 相似度检索小工具：带分数打印 Top-k ==========

# query：自然语言查询；k：返回几条（默认 3）
def search(query, k=3):
    # similarity_search_with_score：返回 (Document, distance/score)
    docs = vectorstore.similarity_search_with_score(query, k=k)
    # 打印查询原文
    print(f"🔍 Query: '{query}'")
    # 逐条展示分数与内容前 50 字
    for i, (doc, score) in enumerate(docs):
        print(f"{i+1}. Score: {score:.3f} - {doc.page_content[:50]}...")
    # 返回完整结果供后续使用
    return docs


In [ ]:
# ========== RAG 问答：检索上下文 → 拼进 system → 调 OpenRouter ==========

# message：用户问题；history：Gradio 兼容参数（本实现未用历史）
def answer_question(message, history=None):
    try:
        # 从 FAISS 取最相关的 3 个块（只要内容，不要分数）
        docs = vectorstore.similarity_search(message, k=3)
        # 把多块正文用空行拼成一段 context
        context = "\n\n".join([d.page_content for d in docs])

        # Chat Completions：system 约束「只根据上下文简答」，user 放原问题
        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"Answer using this context:\n{context}\nBe concise."},
                {"role": "user", "content": message}
            ]
        )
        # 取出助手回复文本
        return response.choices[0].message.content
    except Exception as e:
        # 任何异常都转成可读错误字符串，避免 UI 直接崩
        return f"Error: {str(e)}"

# 冒烟测试：问一个样例员工问题
print("\n🤖 Testing RAG:")
print(answer_question("Who is Alex Lancaster?", []))


In [ ]:
# ========== Gradio ChatInterface：把 answer_question 挂成网页聊天 ==========

# ChatInterface：最简聊天壳；fn 即上面的 RAG 函数
demo = gr.ChatInterface(
    fn=answer_question,
    title="🤖 Insurellm Assistant",
    description="Ask about employees, products, and contracts"
)

# 提示即将启动本地 UI
print("\n🚀 Launching chat...")
# share=False 不生成公网链接；inbrowser=True 尝试自动打开浏览器
demo.launch(share=False, inbrowser=True)


In [ ]:
# ========== 简易性能评估：延迟与回答长度柱状图 ==========

# 三道固定测试题（字符串保持原样，便于对比）
questions = ["Who is Alex Lancaster?", "What is CarLLM?", "Who is the CTO?"]
# times：每题耗时（秒）；lengths：回答字符数
times, lengths = [], []

# 逐题计时调用 RAG
for q in questions:
    # 记录开始时间
    start = time.time()
    # 跑一遍完整问答
    ans = answer_question(q, [])
    # 累加耗时
    times.append(time.time() - start)
    # 记录回答长度
    lengths.append(len(ans))
    # 打印题目前缀与耗时
    print(f"✅ {q[:15]}... → {times[-1]:.2f}s")

# 有数据才画图
if times:
    # 1 行 2 列子图：左耗时、右长度
    fig = make_subplots(rows=1, cols=2)
    # 左：每题耗时柱
    fig.add_trace(go.Bar(x=[f"Q{i+1}" for i in range(len(questions))], y=times), row=1, col=1)
    # 右：每题回答长度柱
    fig.add_trace(go.Bar(x=[f"Q{i+1}" for i in range(len(questions))], y=lengths), row=1, col=2)
    fig.update_layout(title="📈 Performance", height=400)
    fig.show()
    # 平均延迟
    print(f"Avg time: {np.mean(times):.2f}s")


# 第 5 周练习要点（本笔记本）

## 你在练什么
- **RAG**：先检索知识库块，再把上下文塞进 LLM 提示
- **本地嵌入 + FAISS**：不依赖云端 Embedding API 也能建向量索引
- **OpenRouter**：用 OpenAI 兼容客户端调用远端聊天模型
- **Gradio**：几行代码挂出可聊的 UI，并可用 Plotly 看向量分布与延迟

## 怎么跑
1. 配好 `.env` 里的 `OPENROUTER_API_KEY`
2. 从上到下运行：安装导入 → 建库 → 切块嵌入 → 可选可视化 → 聊天 / 测性能
3. 若本地没有 `knowledge-base`，单元格会自动生成 Insurellm 小样例
